# 2. Develop and deploy: source and manifest preparation

**Pre-live authoring draft — not a complete participant lifecycle tutorial.**

Run this notebook from `beginner/` with a Python kernel. All current code cells
are offline and use the standard library. They inspect existing source, render
candidate manifests, and demonstrate versioned image references in temporary
storage. They do not execute JavaScript, run a CLI, build, push or deploy images.

The selected beginner workflow remains participant-executed build → push →
deploy in an organiser-prepared disposable lab. Organiser-mediated deployment
is not a substitute. The protected builder, registry permissions, authentication,
isolation and exact platform/template versions must be resolved and rehearsed
before executable live lifecycle cells can be added.


## 1. Locate source and helpers — offline

**Purpose:** connect function source to the manifest describing its image and route.
**Location:** this notebook's Python kernel, started in `beginner/`.
**Expected result:** the module path and a validated offline template.
**Recovery:** restart the kernel from `beginner/` if the module check fails.


In [ ]:
from pathlib import Path
import json
import sys
import tempfile

MODULE = Path.cwd().resolve().parent
if not (MODULE / "config/lab.example.json").is_file():
    raise RuntimeError("Open this notebook from the module's beginner directory.")
sys.path.insert(0, str(MODULE / "helpers"))
from lab_config import load_config
from manifests import render_manifest, write_manifest, TEMPLATE_REVISION

CONFIG = MODULE / "config/lab.example.json"
config = load_config(CONFIG, require_live=False)
print("Module:", MODULE)
print("Inspected template revision (not enforced by this helper):", TEMPLATE_REVISION)


## 2. Inspect the complete greeting implementation — offline

The developer controls input validation and the successful response. Review the
POST check, object check, name trimming and length bound. The handler does not
install a platform or choose its own resource placement.

**Checkpoint:** locate the response message and the `invalid_name` branch.
The supplied implementation is complete; this inspection is not a TODO exercise.
The package test command requires a separately validated Node.js environment.
Python does not execute or validate the JavaScript shown here.


In [ ]:
print((MODULE / "examples/greeting/handler.js").read_text(encoding="utf-8"))
package = json.loads((MODULE / "examples/greeting/package.json").read_text(encoding="utf-8"))
print("Declared runtime:", package["engines"])
print("Declared unit-test command:", package["scripts"]["test"])


## 3. Render a candidate manifest — offline

**Purpose:** map one owned function to its source, template and versioned image.
**Expected result:** one `group-01-greeting` entry, with `node22` and tag `v1`.
The `.invalid` routes are deliberate placeholders, not a lab you can contact.

The renderer returns JSON-formatted YAML. Acceptance by the pinned CLI and the
absolute handler path remains a separate validation step. The template revision
printed earlier is provenance information: the renderer does not fetch, install
or enforce that revision. Function prefixes check consistency, not permissions.


In [ ]:
manifest_text = render_manifest(CONFIG, "greeting")
manifest = json.loads(manifest_text)
name = config["functions"]["greeting"]
assert set(manifest["functions"]) == {name}
assert manifest["functions"][name]["lang"] == "node22"
assert manifest["functions"][name]["image"].endswith(":" + config["image_tag"])
print(manifest_text)


## 4. Check safe manifest writing — offline, temporary files only

**Purpose:** prevent accidental replacement of an existing manifest.
**Expected result:** the file matches the rendered candidate and a second write
is refused. The temporary directory is removed at the end of the cell.
No local configuration or persistent manifest is created in the repository.


In [ ]:
with tempfile.TemporaryDirectory() as directory:
    candidate = Path(directory) / "greeting.local.json"
    write_manifest(CONFIG, "greeting", candidate)
    assert candidate.read_text(encoding="utf-8") == manifest_text
    try:
        write_manifest(CONFIG, "greeting", candidate)
    except FileExistsError:
        print("PASS: Existing candidate cannot be overwritten.")
    else:
        raise RuntimeError("Existing candidate was unexpectedly overwritten.")
print("Temporary candidate removed.")


## 5. Distinguish request changes from deployed code changes — offline

Sending `{"name": "Ada"}` and then `{"name": "Sam"}` changes request data;
it does not change function code. Changing the response implementation requires
a new image build, push, deployment/update and verification.

Below we change only an image tag in a temporary configuration. This demonstrates
reference construction, not a code update. A tag is a label and may be mutable;
an explicit tag alone does not prove that an image exists or pin its digest.


In [ ]:
updated_config = dict(config)
updated_config["image_tag"] = "v2"
with tempfile.TemporaryDirectory() as directory:
    temporary_config = Path(directory) / "lab.local.json"
    temporary_config.write_text(json.dumps(updated_config), encoding="utf-8")
    updated_manifest = json.loads(render_manifest(temporary_config, "greeting"))
old_image = manifest["functions"][name]["image"]
new_image = updated_manifest["functions"][name]["image"]
assert old_image != new_image
assert new_image.endswith(":v2")
print("Original candidate:", old_image)
print("Updated candidate:", new_image)
print("No source, image, registry or deployed function was changed.")


## 6. Live lifecycle — BLOCKED in this draft

No switch enables these actions here. Complete runnable participant cells will
be added only after the execution contract is documented and tested.

| Participant stage | Required verification before adoption |
|---|---|
| Build | Protected builder access, exact template and architecture, actual image creation and test evidence. Shrinkwrap is not an image build. |
| Push | Correct built-image environment, scoped registry credentials, successful command status and registry evidence. |
| Deploy/update | Supported edition and registry behaviour, assigned-function permissions, correct image reference and negative cross-group tests. |
| Readiness | A validated readiness mechanism with bounded waiting; printed deployment progress is insufficient. |
| Invoke | Verified TLS/access, actual gateway/runtime responses and expected code version. |
| Cleanup | Only owned exercise functions through validated scoped permissions; never remove the prepared function or platform. |

The inspected CLI candidate uses separate build, push and deploy operations.
Its tested push failure does not establish successful registry access. The
inspected CE provider's anonymous-image restriction is an adoption blocker for
any proposed private-image path until actual behaviour and permitted use are
resolved. Do not work around this by publishing secrets or granting unrestricted
registry access. Participants must not receive shared-host Docker sockets or
shared-platform administrator credentials.


## 7. Review and handoff

- Which manifest field points to source, and which points to the runtime image?
- Why does rendering a manifest not demonstrate a successful image build?
- What evidence shows that tag `v2` contains changed code?
- Why must registry permissions and runtime image retrieval both be tested?

**Current limits:** no JavaScript execution or source-update exercise is performed
by this notebook, and no build/push/deploy/readiness commands are implemented.
It is an offline preparation draft, not delivery-ready coverage of these outcomes.
Notebook 3 will explore supplied weather data and bounded-traffic observations;
monitoring and cleanup will retain their live validation gates.
